# Working With Catalogs, Schemas and Volumes
Demonstrates how to create and manage the three-level **Unity Catalog** namespace in Databricks.

| Level | SQL Object | Analogy |
|-------|-----------|----------|
| 1st | **Catalog** | Database server / environment |
| 2nd | **Schema** (Database) | Database / namespace |
| 3rd | **Table / View / Volume** | Table, view, or file storage |

A fully-qualified object name always takes the form: `catalog.schema.object`

**Volumes** are Unity Catalog objects that govern access to non-tabular files stored in cloud object storage.
They sit at the same level as tables and views, inside a schema.

## Setup — Spark Session

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

spark.sql("SELECT current_catalog(), current_schema()").show(truncate=False)

## Catalogs
The top level of the Unity Catalog hierarchy. Each Databricks metastore can hold multiple catalogs.
Use `SHOW CATALOGS` to list all catalogs visible to the current user.

In [ ]:
spark.sql("SHOW CATALOGS").show(truncate=False)

## Schemas (Databases)
Schemas are the second level of the hierarchy — they live inside a catalog and group related tables,
views, and volumes together. `CREATE SCHEMA` and `CREATE DATABASE` are synonymous in Spark SQL.

We will create a new schema **`country_analytics`** inside the current catalog to isolate our work.

In [ ]:
# Resolve the active catalog dynamically so the notebook is portable
catalog = spark.sql("SELECT current_catalog() AS c").collect()[0]["c"]
new_schema = "country_analytics"

spark.sql(f"SHOW SCHEMAS IN `{catalog}`").show(truncate=False)

In [ ]:
catalog = spark.sql("SELECT current_catalog() AS c").collect()[0]["c"]
new_schema = "country_analytics"

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{new_schema}`
    COMMENT 'Analytics layer for country population data'
""")

print(f"Schema created: {catalog}.{new_schema}")

# Verify — the new schema should now appear in the list
spark.sql(f"SHOW SCHEMAS IN `{catalog}`").show(truncate=False)

In [ ]:
catalog = spark.sql("SELECT current_catalog() AS c").collect()[0]["c"]
new_schema = "country_analytics"

# DESCRIBE SCHEMA EXTENDED shows location, owner, comment and properties
spark.sql(f"DESCRIBE SCHEMA EXTENDED `{catalog}`.`{new_schema}`").show(truncate=False)

## Volumes
A **Volume** is a Unity Catalog object that provides governed access to files in cloud object storage
(e.g., CSV, JSON, Parquet, images) without treating them as tabular data.

| Volume Type | Storage | Use Case |
|-------------|---------|----------|
| **Managed** | Databricks-managed storage | Landing zones, staging files |
| **External** | Customer-owned storage | Existing data lake paths |

Files inside a volume are accessible via the `/Volumes/catalog/schema/volume/` path prefix.

In [ ]:
catalog = spark.sql("SELECT current_catalog() AS c").collect()[0]["c"]
new_schema = "country_analytics"
volume_name = "raw_files"

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS `{catalog}`.`{new_schema}`.`{volume_name}`
    COMMENT 'Managed volume for raw input files in the country_analytics schema'
""")

print(f"Volume created: {catalog}.{new_schema}.{volume_name}")
print(f"Volume path  : /Volumes/{catalog}/{new_schema}/{volume_name}/")

In [ ]:
catalog = spark.sql("SELECT current_catalog() AS c").collect()[0]["c"]
new_schema = "country_analytics"

# SHOW VOLUMES lists all volumes in the schema
spark.sql(f"SHOW VOLUMES IN `{catalog}`.`{new_schema}`").show(truncate=False)

## Populating a Table from an Existing Source
We read from the `country_population` table created in the **Managed Tables** notebook
(`<current catalog>.default.country_population`) and write
into two new Delta tables inside `country_analytics` using `spark.sql`.

| Target Table | Description |
|---|---|
| `country_population_all` | Full copy of the source table |
| `country_population_large` | Filtered — countries with population > 50 million |

In [ ]:
catalog = spark.sql("SELECT current_catalog() AS c").collect()[0]["c"]
new_schema = "country_analytics"
source_table = f"`{catalog}`.`default`.`country_population`"

# ── Full copy (CTAS) ──────────────────────────────────────────────────────────
spark.sql(f"""
    CREATE OR REPLACE TABLE `{catalog}`.`{new_schema}`.`country_population_all`
    COMMENT 'Full copy of the country_population source table'
    AS
    SELECT *
    FROM {source_table}
""")

count = spark.sql(f"SELECT COUNT(*) AS n FROM `{catalog}`.`{new_schema}`.`country_population_all`").collect()[0]["n"]
print(f"country_population_all  — {count} rows")

In [ ]:
catalog = spark.sql("SELECT current_catalog() AS c").collect()[0]["c"]
new_schema = "country_analytics"
source_table = f"`{catalog}`.`default`.`country_population`"

# ── Filtered subset ───────────────────────────────────────────────────────────
spark.sql(f"""
    CREATE OR REPLACE TABLE `{catalog}`.`{new_schema}`.`country_population_large`
    COMMENT 'Countries with population greater than 50 million'
    AS
    SELECT *
    FROM {source_table}
    WHERE POPULATION > 50000000
    ORDER BY POPULATION DESC
""")

count = spark.sql(f"SELECT COUNT(*) AS n FROM `{catalog}`.`{new_schema}`.`country_population_large`").collect()[0]["n"]
print(f"country_population_large — {count} rows")

spark.sql(f"""
    SELECT *
    FROM `{catalog}`.`{new_schema}`.`country_population_large`
""").show(truncate=False)

## Inspecting the Schema
After creating tables, use `SHOW TABLES` to confirm what objects exist in the schema.

In [ ]:
catalog = spark.sql("SELECT current_catalog() AS c").collect()[0]["c"]
new_schema = "country_analytics"

print("=== Tables ===")
spark.sql(f"SHOW TABLES IN `{catalog}`.`{new_schema}`").show(truncate=False)

print("=== Volumes ===")
spark.sql(f"SHOW VOLUMES IN `{catalog}`.`{new_schema}`").show(truncate=False)

In [ ]:
catalog = spark.sql("SELECT current_catalog() AS c").collect()[0]["c"]
new_schema = "country_analytics"

# DESCRIBE EXTENDED confirms Type=MANAGED and Provider=delta
spark.sql(f"""
    DESCRIBE EXTENDED `{catalog}`.`{new_schema}`.`country_population_large`
""").filter("col_name IN ('Type', 'Provider', 'Location', 'Comment')").show(truncate=False)

## Cleanup
`DROP SCHEMA ... CASCADE` removes the schema together with **all** of its tables, views, and volumes.
Without `CASCADE` the command fails if the schema is non-empty.

In [ ]:
catalog = spark.sql("SELECT current_catalog() AS c").collect()[0]["c"]
new_schema = "country_analytics"

spark.sql(f"DROP SCHEMA IF EXISTS `{catalog}`.`{new_schema}` CASCADE")

print(f"Schema '{new_schema}' and all its contents dropped.")

# Verify — country_analytics should no longer appear
spark.sql(f"SHOW SCHEMAS IN `{catalog}`").show(truncate=False)